In [1]:
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time
import random
import os
import re # Biblioteca para identificar padrões em textos (como números)

def extrair_salario_para_int(texto):
    """
    Função que pega num texto sujo e devolve apenas o número inteiro.
    Se não achar dinheiro, devolve 0.
    """
    texto = str(texto).lower()
    
    if "não informado" in texto or "indisponível" in texto or texto == "nan":
        return 0
    
    # Regex: Procura qualquer bloco que contenha números e pontos (ex: 5.000)
    numeros = re.findall(r'[\d\.]+', texto)
    
    if numeros:
        # Pega a primeira ocorrência encontrada
        primeiro_numero = numeros[0]
        # Tira o ponto de milhar (5.000 vira 5000)
        primeiro_numero_limpo = primeiro_numero.replace('.', '')
        
        try:
            return int(primeiro_numero_limpo)
        except:
            return 0
            
    return 0

def main():
    print("A iniciar o Processador de Dados...")
    
    # 1. Carrega o ficheiro de dados que já tem
    try:
        df = pd.read_csv("vagas_adzuna_completas_checkpoint.csv")
        
        # <-- CORREÇÃO 1: Limpar o índice para evitar o erro de colunas incompatíveis -->
        df = df.reset_index(drop=True) 
        
        print(f"Sucesso! {len(df)} vagas carregadas para processamento.")
    except FileNotFoundError:
        print("Erro: O ficheiro 'vagas_adzuna_completas_checkpoint.csv' não foi encontrado na pasta.")
        return

    # 2. TRANSFORMAÇÃO DO SALÁRIO (Rápido, feito na memória)
    print("\nA limpar e a padronizar a coluna de Salários para números Inteiros (0 = Não informado)...")
    df['Salario_Int'] = df['Salario'].apply(extrair_salario_para_int)
    print("Salários padronizados com sucesso!")
    
    # Prepara a coluna de data (Se não existir, ele cria)
    if 'Data_Publicacao' not in df.columns:
        df['Data_Publicacao'] = "Pendente"
    
    # 3. EXTRAÇÃO DAS DATAS NA INTERNET (Demorado)
    print("\nA iniciar o robô para buscar as datas nas páginas...")
    driver = uc.Chrome()
    
    for indice, linha in df.iterrows():
        # SISTEMA DE RETOMADA: Se a data já foi extraída num backup anterior, ele pula e ganha tempo!
        if linha['Data_Publicacao'] != "Pendente":
            continue
            
        link = str(linha['Link'])
        if not link.startswith("http"):
            continue
            
        print(f"[{indice + 1}/{len(df)}] A procurar data em: {str(linha['Titulo'])[:30]}...")
        
        try:
            driver.get(link)
            time.sleep(random.uniform(3, 6)) # Pausa antibot
            
            # TENTA ACHAR A DATA:
            try:
                elemento_data = driver.find_element(By.XPATH, "//*[contains(text(), 'Publicado') or contains(text(), 'dias atrás') or contains(text(), 'Há')]")
                data_texto = elemento_data.text.strip()
            except:
                data_texto = "Data não encontrada na página"
                
            # <-- CORREÇÃO 2: Guarda o texto da data usando .loc e forçando a conversão para string -->
            df.loc[indice, 'Data_Publicacao'] = str(data_texto)
            print(f"-> {data_texto}")
            
        except Exception as e:
            print("-> Erro ao aceder à página. A avançar para a próxima...")
            # <-- CORREÇÃO 2: Usando .loc também na parte do erro -->
            df.loc[indice, 'Data_Publicacao'] = "Erro de conexão"
            
        # SALVAMENTO DE SEGURANÇA: Guarda o ficheiro a cada 10 vagas lidas
        if (indice + 1) % 10 == 0:
            df.to_csv("vagas_adzuna_completas_checkpoint.csv", index=False, encoding="utf-8-sig")

    # Salvamento Final
    # O ficheiro final terá as colunas antigas + a coluna "Salario_Int" + a coluna "Data_Publicacao"
    df.to_csv("Base_Final_Pronta_Pro_PowerBI.csv", index=False, encoding="utf-8-sig")
    print("\nProcesso 100% concluído! O ficheiro 'Base_Final_Pronta_Pro_PowerBI.csv' foi gerado.")
    
    driver.quit()
    os._exit(0)

if __name__ == '__main__':
    main()

A iniciar o Processador de Dados...
Erro: O ficheiro 'vagas_adzuna_completas_checkpoint.csv' não foi encontrado na pasta.
